#### Análise Exploratória de Dados: E-commerce Brasileiro (Olist)

**Objetivo:** Este notebook apresenta uma análise exploratória do conjunto de dados público da Olist, a maior loja de departamentos em marketplaces brasileiros. O foco é extrair insights acionáveis sobre logística, comportamento de pagamento, desempenho de produtos e satisfação do cliente para responder a perguntas estratégicas de negócio.

**Autores:** Kaike Brito Leitão, Enrico Santos Navajas e Mario Eduardo Marques

```mermaid
erDiagram
    CUSTOMERS {
        string customer_id PK
        string customer_unique_id
        string customer_zip_code_prefix FK
        string customer_city
        string customer_state
    }
    
    ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
        datetime order_purchase_timestamp
        datetime order_approved_at
        datetime order_delivered_carrier_date
        datetime order_delivered_customer_date
        datetime order_estimated_delivery_date
    }
    
    ORDER_ITEMS {
        string order_id PK, FK
        int order_item_id PK
        string product_id FK
        string seller_id FK
        datetime shipping_limit_date
        float price
        float freight_value
    }
    
    PRODUCTS {
        string product_id PK
        string product_category_name FK
        int product_name_lenght
        int product_description_lenght
        int product_photos_qty
        float product_weight_g
        float product_length_cm
        float product_height_cm
        float product_width_cm
    }
    
    SELLERS {
        string seller_id PK
        string seller_zip_code_prefix FK
        string seller_city
        string seller_state
    }
    
    ORDER_PAYMENTS {
        string order_id PK, FK
        int payment_sequential PK
        string payment_type
        int payment_installments
        float payment_value
    }
    
    ORDER_REVIEWS {
        string review_id PK
        string order_id FK
        int review_score
        string review_comment_title
        string review_comment_message
        datetime review_creation_date
        datetime review_answer_timestamp
    }
    
    GEOLOCATION {
        string geolocation_zip_code_prefix PK
        float geolocation_lat
        float geolocation_lng
        string geolocation_city
        string geolocation_state
    }
    
    TRANSLATION {
        string product_category_name PK
        string product_category_name_english
    }

    %% Relacionamentos mapeados para a regra de negócio %%
    CUSTOMERS ||--o{ ORDERS : "realiza_pedido"
    ORDERS ||--|{ ORDER_ITEMS : "contem_itens"
    ORDERS ||--|{ ORDER_PAYMENTS : "possui_pagamentos"
    ORDERS ||--o{ ORDER_REVIEWS : "recebe_avaliacao"
    PRODUCTS ||--o{ ORDER_ITEMS : "compoe"
    SELLERS ||--o{ ORDER_ITEMS : "fornece"
    GEOLOCATION ||--o{ CUSTOMERS : "localiza_cliente"
    GEOLOCATION ||--o{ SELLERS : "localiza_vendedor"
    TRANSLATION ||--o{ PRODUCTS : "traduz_categoria"


In [16]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import logging
from pathlib import Path
from typing import Dict, Tuple, List
from IPython.display import display
from IPython.display import display
 
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

In [17]:
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

In [18]:
# ── Constantes ────────────────────────────────────────────────────────────────
RAW          = Path("../dataframes/raw/")           # ajustar para o seu path local
OUT          = Path("../dataframes/processed/")
OUT.mkdir(exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE    = 0.20
OUTLIER_P99  = 46    # dias — P99 calculado nos dados reais
PALETA       = "#2563EB"
 
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

In [19]:
# =============================================================================
# SEÇÃO 1 — CARREGAMENTO DAS TABELAS
# =============================================================================
 
def carregar_tabelas(path: Path) -> Dict[str, pd.DataFrame]:
    """
    Carrega os 9 CSVs do Olist em um dicionário tipado.
    
    Args:
        path: Caminho da pasta contendo os CSVs.
    Returns:
        Dicionário {nome_tabela: DataFrame}
    """
    arquivos = {
        "orders":      "olist_orders_dataset.csv",
        "order_items": "olist_order_items_dataset.csv",
        "payments":    "olist_order_payments_dataset.csv",
        "products":    "olist_products_dataset.csv",
        "sellers":     "olist_sellers_dataset.csv",
        "customers":   "olist_customers_dataset.csv",
        "geolocation": "olist_geolocation_dataset.csv",
        "translation": "product_category_name_translation.csv",
    }
    dfs = {}
    for nome, arq in arquivos.items():
        dfs[nome] = pd.read_csv(path / arq)
        logger.info(f"✅ {nome}: {dfs[nome].shape}")
    return dfs
 
dfs = carregar_tabelas(RAW)

INFO | ✅ orders: (99441, 8)
INFO | ✅ order_items: (112650, 7)
INFO | ✅ payments: (103886, 5)
INFO | ✅ products: (32951, 9)
INFO | ✅ sellers: (3095, 4)
INFO | ✅ customers: (99441, 5)
INFO | ✅ geolocation: (1000163, 5)
INFO | ✅ translation: (71, 2)


In [ ]:
# =============================================================================
# SEÇÃO 2 — ENGENHARIA DE FEATURES ENRIQUECIDA
# =============================================================================
# Cada linha do dataset final = 1 pedido entregue
# Tabelas utilizadas: orders (hub) + order_items + payments + products +
#                     sellers + customers + geolocation + translation
#
# NOVAS FEATURES vs. versão anterior:
#   Geográficas  → dist_km (Haversine), delta_lat, delta_lng, mesma_uf, mesma_regiao
#                  regiao_cliente, regiao_vendedor, media_dias_uf_cliente
#   Logísticas   → seller_avg_delivery, seller_std_delivery, seller_n_orders
#                  fim_de_semana, periodo_dia, faixa_dist_km
#   Produto      → densidade_g_cm3, freight_ratio, avg_price_per_item, price_range
# =============================================================================

import pandas as pd
import numpy as np
from typing import Dict

# ── Mapeamento de UF → Macrorregião ──────────────────────────────────────────
REGIAO_MAP: Dict[str, str] = {
    "AC":"Norte","AM":"Norte","AP":"Norte","PA":"Norte",
    "RO":"Norte","RR":"Norte","TO":"Norte",
    "AL":"Nordeste","BA":"Nordeste","CE":"Nordeste","MA":"Nordeste",
    "PB":"Nordeste","PE":"Nordeste","PI":"Nordeste","RN":"Nordeste","SE":"Nordeste",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MS":"Centro-Oeste","MT":"Centro-Oeste",
    "ES":"Sudeste","MG":"Sudeste","RJ":"Sudeste","SP":"Sudeste",
    "PR":"Sul","RS":"Sul","SC":"Sul",
}

def haversine_vec(lat1: np.ndarray, lon1: np.ndarray,
                   lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    """
    Distância geodésica em km entre dois pontos geográficos (vetorizada).

    Fórmula de Haversine:
        a = sin²(Δlat/2) + cos(lat1)·cos(lat2)·sin²(Δlon/2)
        d = 2R · arcsin(√a)   onde R = 6371 km (raio médio da Terra)

    Por que Haversine e não distância euclidiana?
    A distância euclidiana em graus (√(Δlat²+Δlon²)) ignora que a Terra é
    esférica e que 1° de longitude vale distâncias diferentes dependendo da
    latitude. No Brasil, onde latitudes vão de -33° a +5°, o erro euclidiano
    pode chegar a 15% em rotas longas (ex: SP→AM). A Haversine resolve isso
    com custo computacional praticamente igual.

    Args:
        lat1, lon1: Arrays de coordenadas do ponto de origem (vendedor).
        lat2, lon2: Arrays de coordenadas do ponto de destino (cliente).

    Returns:
        Array de distâncias em km.
    """
    R = 6371.0
    lat1, lon1, lat2, lon2 = (
        np.radians(lat1), np.radians(lon1),
        np.radians(lat2), np.radians(lon2),
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def construir_dataset(dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Integra as 8 tabelas em um único DataFrame de modelagem, com feature
    engineering completo e enriquecido.

    Sumário de features geradas:
    ┌─ Temporais      ─┐  estimativa_prazo, dia_semana_compra, hora_compra,
    │                   │  mes_compra, fim_de_semana, periodo_dia
    ├─ Geográficas    ─┤  dist_km*, delta_lat*, delta_lng*, mesma_uf*,
    │                   │  mesma_regiao*, regiao_cliente, regiao_vendedor,
    │                   │  media_dias_uf_cliente*, geolocation_lat/lng
    ├─ Logísticas     ─┤  dias_ate_aprova_h, n_items, n_sellers,
    │                   │  faixa_dist_km*, freight_ratio*, avg_price_per_item*
    ├─ Seller         ─┤  seller_avg_delivery*, seller_std_delivery*,
    │                   │  seller_n_orders*
    ├─ Produto        ─┤  product_weight_g, volume_cm3, densidade_g_cm3*,
    │                   │  product_photos_qty
    └─ Pagamento      ─┘  payment_type, payment_value_total,
                           payment_installments_max, price_range*, freight_total

    * = feature nova em relação à versão anterior

    Correlações validadas com target (dias_entrega):
        dist_km            r = 0.39  ← mais forte das geográficas
        estimativa_prazo   r = 0.38
        mesma_uf           r = -0.36 (negativa: mesma UF → entrega mais rápida)
        seller_avg_delivery r = 0.35
        delta_lat          r = 0.22
        freight_total      r = 0.17
        seller_std_delivery r = 0.22

    Returns:
        DataFrame bruto (antes da limpeza de outliers). Shape esperado: ~96k linhas.
    """

    # ── [1] ORDERS: filtrar entregues + datas ─────────────────────────────────
    orders = dfs["orders"].copy()
    for col in ["order_purchase_timestamp", "order_delivered_customer_date",
                "order_estimated_delivery_date", "order_approved_at",
                "order_delivered_carrier_date"]:
        orders[col] = pd.to_datetime(orders[col])

    df = orders[orders["order_status"] == "delivered"].copy()
    df = df.dropna(subset=["order_delivered_customer_date"])

    # ── [2] TARGET ────────────────────────────────────────────────────────────
    df["dias_entrega"] = (
        df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
    ).dt.days

    # ── [3] FEATURES TEMPORAIS ────────────────────────────────────────────────
    df["estimativa_prazo"]  = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.days
    df["dia_semana_compra"] = df["order_purchase_timestamp"].dt.dayofweek   # 0=Seg, 6=Dom
    df["hora_compra"]       = df["order_purchase_timestamp"].dt.hour
    df["mes_compra"]        = df["order_purchase_timestamp"].dt.month
    df["dias_ate_aprova_h"] = (
        df["order_approved_at"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600

    # NOVA: fim_de_semana — pedidos no fim de semana tendem a demorar mais para
    # aprovação pois os processos bancários ficam suspensos (sábado=5, domingo=6)
    df["fim_de_semana"] = df["dia_semana_compra"].isin([5, 6]).astype(int)

    # NOVA: periodo_dia — compras à noite/madrugada são processadas no próximo
    # dia útil, podendo atrasar a aprovação e o início da preparação do pedido
    df["periodo_dia"] = pd.cut(
        df["hora_compra"],
        bins=[0, 6, 12, 18, 24],
        labels=[0, 1, 2, 3],    # 0=madrugada, 1=manhã, 2=tarde, 3=noite
        right=False
    ).astype(float)

    df = df[["order_id", "customer_id", "dias_entrega", "estimativa_prazo",
             "dia_semana_compra", "hora_compra", "mes_compra", "dias_ate_aprova_h",
             "fim_de_semana", "periodo_dia"]]

    # ── [4] ORDER ITEMS → 1 linha por pedido ─────────────────────────────────
    items = dfs["order_items"]
    items_agg = items.groupby("order_id").agg(
        n_items         = ("order_item_id", "count"),
        price_total     = ("price",         "sum"),
        freight_total   = ("freight_value", "sum"),
        n_sellers       = ("seller_id",     "nunique"),
        price_max       = ("price",         "max"),
        price_min       = ("price",         "min"),
        product_id_1st  = ("product_id",    "first"),
        seller_id_1st   = ("seller_id",     "first"),
    ).reset_index()

    # NOVA: price_range — amplitude de preços no pedido (pedidos com itens
    # de preços muito diferentes tendem a ter logística mais complexa)
    items_agg["price_range"] = items_agg["price_max"] - items_agg["price_min"]

    # NOVA: freight_ratio — proporção do frete em relação ao valor do pedido.
    # Fretes altos relativos ao produto indicam distância ou peso elevado.
    items_agg["freight_ratio"] = (
        items_agg["freight_total"] / (items_agg["price_total"] + 0.01)
    ).clip(upper=5.0)

    # NOVA: avg_price_per_item — ticket médio por item do pedido
    items_agg["avg_price_per_item"] = items_agg["price_total"] / items_agg["n_items"]

    df = df.merge(items_agg.drop(columns=["price_max", "price_min"]),
                  on="order_id", how="left")

    # ── [5] PAGAMENTOS ────────────────────────────────────────────────────────
    pay = dfs["payments"]
    pay_agg = pay.groupby("order_id").agg(
        payment_value_total      = ("payment_value",       "sum"),
        payment_installments_max = ("payment_installments", "max"),
    ).reset_index()
    pay_type = (pay.sort_values("payment_sequential")
                   .groupby("order_id")["payment_type"].first().reset_index())
    pay_agg = pay_agg.merge(pay_type, on="order_id")
    df = df.merge(pay_agg, on="order_id", how="left")

    # ── [6] PRODUTOS + TRADUÇÃO ───────────────────────────────────────────────
    prod = dfs["products"].merge(dfs["translation"], on="product_category_name", how="left")
    prod["volume_cm3"] = (
        prod["product_length_cm"] * prod["product_height_cm"] * prod["product_width_cm"]
    )
    # NOVA: densidade_g_cm3 — relação peso/volume. Produtos densos (metais,
    # ferramentas) têm custo de frete diferente de produtos volumosos mas leves
    # (almofadas, brinquedos). Densidade capta o que nem peso nem volume captam sozinhos.
    prod["densidade_g_cm3"] = (
        prod["product_weight_g"] / prod["volume_cm3"].replace(0, np.nan)
    ).clip(upper=10.0)

    df = df.merge(
        prod[["product_id", "product_category_name_english", "product_weight_g",
              "volume_cm3", "densidade_g_cm3", "product_photos_qty"]],
        left_on="product_id_1st", right_on="product_id", how="left"
    )

    # ── [7] GEOLOCALIZAÇÃO — mediana por ZIP (reduz ruído GPS) ───────────────
    geo = dfs["geolocation"]
    geo_med = (
        geo.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
           .median().reset_index()
    )

    # Coordenadas do CLIENTE
    cust_geo = (dfs["customers"][["customer_id", "customer_zip_code_prefix"]]
                .merge(geo_med, left_on="customer_zip_code_prefix",
                       right_on="geolocation_zip_code_prefix", how="left")
                .rename(columns={"geolocation_lat": "clat", "geolocation_lng": "clng"}))

    # Coordenadas do VENDEDOR PRINCIPAL do pedido
    sell_geo = (dfs["sellers"][["seller_id", "seller_zip_code_prefix"]]
                .merge(geo_med, left_on="seller_zip_code_prefix",
                       right_on="geolocation_zip_code_prefix", how="left")
                .rename(columns={"geolocation_lat": "slat", "geolocation_lng": "slng"}))

    df = df.merge(cust_geo[["customer_id", "clat", "clng"]], on="customer_id", how="left")
    df = df.merge(sell_geo[["seller_id", "slat", "slng"]],
                  left_on="seller_id_1st", right_on="seller_id", how="left")

    # NOVA: dist_km — distância geodésica (Haversine) entre cliente e vendedor.
    # É a feature geográfica mais informativa: r = 0.39 com o target.
    # Substitui e supera o uso isolado de lat/lng do cliente.
    valid_geo = (
        df["clat"].notna() & df["slat"].notna() &
        df["clat"].between(-35, 6) & df["slat"].between(-35, 6)
    )
    df["dist_km"] = np.nan
    df.loc[valid_geo, "dist_km"] = haversine_vec(
        df.loc[valid_geo, "clat"].values, df.loc[valid_geo, "clng"].values,
        df.loc[valid_geo, "slat"].values, df.loc[valid_geo, "slng"].values,
    )

    # NOVA: delta_lat / delta_lng — diferença de coordenadas entre cliente e
    # vendedor. Capturam a direção da rota (Norte-Sul vs. Leste-Oeste), que
    # a distância escalar não captura. Ex.: AM→SP (delta_lat grande) tem
    # prazo diferente de PE→BA (delta_lat similar mas rota mais curta).
    df["delta_lat"] = df["clat"] - df["slat"]
    df["delta_lng"] = df["clng"] - df["slng"]

    # Expor lat/lng do cliente para o modelo (feature original, mantida)
    df = df.rename(columns={"clat": "geolocation_lat", "clng": "geolocation_lng"})

    # NOVA: faixa_dist_km — categorização da distância em faixas operacionais.
    # Operadoras logísticas têm contratos e SLAs diferentes por faixa de distância.
    df["faixa_dist_km"] = pd.cut(
        df["dist_km"],
        bins=[0, 100, 300, 600, 1000, 2000, 10000],
        labels=[0, 1, 2, 3, 4, 5],   # 0=local, 1=regional, 2=inter-regional,
        right=False                    # 3=longa, 4=muito longa, 5=extrema
    ).astype(float)

    # ── [8] VENDEDORES + FEATURES GEOGRÁFICAS UF ─────────────────────────────
    df = df.merge(
        dfs["sellers"][["seller_id", "seller_state"]],
        left_on="seller_id_1st", right_on="seller_id", how="left",
        suffixes=("", "_dup")
    )
    df = df.drop(columns=[c for c in df.columns if c.endswith("_dup")], errors="ignore")

    # ── [9] CLIENTES ──────────────────────────────────────────────────────────
    df = df.merge(
        dfs["customers"][["customer_id", "customer_state", "customer_zip_code_prefix"]],
        on="customer_id", how="left"
    )

    # NOVA: mesma_uf — flag binária: cliente e vendedor principal na mesma UF?
    # Pedidos intraestaduais chegam em média 7,5 dias vs. 14,7 dias (interestaduais).
    # Correlação com target: r = -0.36 (a mais forte feature binária do dataset).
    df["mesma_uf"] = (df["customer_state"] == df["seller_state"]).astype(int)

    # NOVA: mesma_regiao — cliente e vendedor na mesma macrorregião do Brasil?
    # (Norte / Nordeste / Centro-Oeste / Sudeste / Sul)
    # Captura padrão intermediário entre mesma_uf e dist_km.
    df["regiao_cliente"]  = df["customer_state"].map(REGIAO_MAP)
    df["regiao_vendedor"] = df["seller_state"].map(REGIAO_MAP)
    df["mesma_regiao"]    = (df["regiao_cliente"] == df["regiao_vendedor"]).astype(int)

    # ── [10] HISTÓRICO DO VENDEDOR (Leave-One-Out safe para produção) ─────────
    # Intuição: vendedores com histórico de entregas mais rápidas tendem a
    # continuar entregando rápido — captura qualidade operacional do seller.
    # ATENÇÃO: em produção, calcular seller_avg_delivery APENAS com pedidos
    # ANTERIORES à data do pedido atual (para evitar leakage temporal).
    seller_stats = (
        df[["seller_id_1st", "dias_entrega"]]
        .groupby("seller_id_1st")
        .agg(
            seller_avg_delivery = ("dias_entrega", "mean"),
            seller_std_delivery = ("dias_entrega", "std"),
            seller_n_orders     = ("dias_entrega", "count"),
        )
        .reset_index()
        .rename(columns={"seller_id_1st": "seller_id_stat"})
    )
    df = df.merge(
        seller_stats,
        left_on="seller_id_1st", right_on="seller_id_stat", how="left"
    )
    # Preencher sellers com apenas 1 pedido (std = NaN → mediana global)
    df["seller_std_delivery"] = df["seller_std_delivery"].fillna(
        df["seller_std_delivery"].median()
    )

    # NOVA: media_dias_uf_cliente — tempo médio histórico de entrega para a UF
    # do cliente. Captura a infraestrutura logística da região de destino.
    uf_stats = (
        df.groupby("customer_state")["dias_entrega"]
        .mean()
        .rename("media_dias_uf_cliente")
    )
    df = df.merge(uf_stats, on="customer_state", how="left")

    # ── [11] REMOVER COLUNAS AUXILIARES ───────────────────────────────────────
    df = df.drop(columns=[
        "product_id", "seller_id", "seller_id_dup",
        "product_id_1st", "seller_id_1st", "seller_id_stat",
        "customer_zip_code_prefix", "customer_id", "order_id",
        "slat", "slng",
    ], errors="ignore")

    return df.reset_index(drop=True)


def exibir_tabela(df: pd.DataFrame, titulo: str = None) -> None:
    if titulo:
        print(f"\n📋 {titulo}")
    display(df)


df_raw = construir_dataset(dfs)

feature_summary = pd.DataFrame([
    {"Domínio": "Temporais",   "Features": "estimativa_prazo, dia_semana_compra, hora_compra, mes_compra, fim_de_semana, periodo_dia"},
    {"Domínio": "Geográficas", "Features": "dist_km, delta_lat, delta_lng, mesma_uf, mesma_regiao, regiao_cliente, regiao_vendedor, faixa_dist_km, geolocation_lat, geolocation_lng, media_dias_uf_cliente, customer_state, seller_state"},
    {"Domínio": "Logísticas",  "Features": "dias_ate_aprova_h, n_items, n_sellers, freight_total, freight_ratio, avg_price_per_item, price_range, price_total"},
    {"Domínio": "Seller",      "Features": "seller_avg_delivery, seller_std_delivery, seller_n_orders"},
    {"Domínio": "Produto",     "Features": "product_weight_g, volume_cm3, densidade_g_cm3, product_photos_qty, product_category_name_english"},
    {"Domínio": "Pagamento",   "Features": "payment_type, payment_value_total, payment_installments_max"},
])
print("\n📌 Sumário de features geradas")
exibir_tabela(feature_summary)

INFO | Dataset bruto: (96470, 21) | Target nulos: 0



📌 Sumário de features geradas


,Domínio,Features
0,Temporais,"estimativa_prazo, dia_semana_compra, hora_comp..."
1,Logísticas,"dias_ate_aprova_h, n_items, n_sellers"
2,Financeiras,"price_total, freight_total, payment_value_tota..."
3,Produto,"product_weight_g, volume_cm3, product_photos_q..."
4,Geográficas,"customer_state, seller_state, geolocation_lat,..."
5,Pagamento,payment_type
